# 02 — Cleaning & Wrangling

**Stage 2 of the Tier A pipeline.** Reads the immutable copy from `01`, never the archive. Every change is
logged with the number of rows it touched and why. Nothing is imputed and no row is dropped: the file has
no errors to remove, only structure to make explicit.

| Carried from `01` | Done here |
|---|---|
| Form labels are glued onto names | §2.2 split into `species` + `form`, `form_kind` ∈ {base, mega, alt} |
| 800 rows ≠ 721 species | §2.2 `is_species` marks the analysis grain |
| Legendary flag mixes two definitions | §2.3 `legendary_canon`, raw flag kept |
| Evolution families break independence | §2.4 `family` proxy for grouped validation |
| Review §7 derived columns | §2.5 shares, bulk, ratios, spread, tiers |

In [1]:
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

ROOT = Path.cwd()
while not (ROOT / "archive" / "Pokemon.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ARCHIVE = ROOT / "archive"
RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"
REPORTS, FIGS, MODELS = ROOT / "reports", ROOT / "reports" / "figures", ROOT / "models"
for p in (RAW, PROC, REPORTS, FIGS, MODELS):
    p.mkdir(parents=True, exist_ok=True)

KF_PATH = REPORTS / "_key_figures.json"
STATS = ["hp", "atk", "def", "spa", "spd", "spe"]
STAT_LABEL = {"hp": "HP", "atk": "Attack", "def": "Defense", "spa": "Sp. Atk", "spd": "Sp. Def", "spe": "Speed"}
SHARES = [f"sh_{s}" for s in STATS]


def _plain(v):
    if isinstance(v, dict):
        return {str(k): _plain(x) for k, x in v.items()}
    if hasattr(v, "item"):
        return v.item()
    if isinstance(v, (list, tuple)):
        return [_plain(x) for x in v]
    return v


def kf_update(**kw):
    """Append to reports/_key_figures.json — the only place the report reads numbers from."""
    kf = json.loads(KF_PATH.read_text(encoding="utf-8")) if KF_PATH.exists() else {}
    kf.update({k: _plain(v) for k, v in kw.items()})
    KF_PATH.write_text(json.dumps(kf, indent=1, sort_keys=True, ensure_ascii=False), encoding="utf-8")
    print(f"  _key_figures.json <- {', '.join(kw)}  ({len(kf)} figures total)")

import re

raw = pd.read_csv(RAW / "pokemon_raw_immutable.csv", encoding="utf-8")
log = []


def note(step, rows, why):
    log.append({"step": step, "rows_affected": int(rows), "why": why})
    print(f"[{len(log):02d}] {step}: {rows} rows — {why}")


df = raw.rename(columns={"#": "dex", "Name": "name_raw", "Type 1": "type1", "Type 2": "type2", "Total": "total",
                         "HP": "hp", "Attack": "atk", "Defense": "def", "Sp. Atk": "spa", "Sp. Def": "spd",
                         "Speed": "spe", "Generation": "generation", "Legendary": "legendary_raw"})
note("rename to snake_case", len(df), "stable, quote-free column names")
print(df.shape)

[01] rename to snake_case: 800 rows — stable, quote-free column names
(800, 13)


### 2.1 Types and dtypes

`type2` is blank for 386 rows. That is a **structural absence** (the creature has one type), not missing
data, so it is not imputed. It stays null and `is_dual_type` records it. Types become a categorical with
the 18 canonical levels, so every group-by returns all 18 in a fixed order.

In [2]:
TYPE_ORDER = ["Normal", "Fire", "Water", "Grass", "Electric", "Ice", "Fighting", "Poison", "Ground", "Flying",
              "Psychic", "Bug", "Rock", "Ghost", "Dragon", "Dark", "Steel", "Fairy"]
for c in ("type1", "type2"):
    df[c] = pd.Categorical(df[c], categories=TYPE_ORDER)
df["is_dual_type"] = df["type2"].notna()
df["type_combo"] = np.where(df.is_dual_type, df.type1.astype(str) + "/" + df.type2.astype(str), df.type1.astype(str))
for c in STATS + ["total", "generation", "dex"]:
    df[c] = df[c].astype("int64")
note("type1/type2 -> 18-level categorical", len(df), "fixed order, all levels present in every group-by")
note("type2 left null where single-typed", int((~df.is_dual_type).sum()), "structural absence (one type), not missing data — no imputation")
print(df.dtypes.to_string())

[02] type1/type2 -> 18-level categorical: 800 rows — fixed order, all levels present in every group-by
[03] type2 left null where single-typed: 386 rows — structural absence (one type), not missing data — no imputation
dex                 int64
name_raw              str
type1            category
type2            category
total               int64
hp                  int64
atk                 int64
def                 int64
spa                 int64
spd                 int64
spe                 int64
generation          int64
legendary_raw        bool
is_dual_type         bool
type_combo            str


### 2.2 Split glued form names; set the analysis grain

The form label follows the species name with no separator (`VenusaurMega Venusaur`, `DeoxysNormal Forme`,
`MeowsticMale`). The split point is the first lower-case letter followed directly by a capital. Names such as
`Mr. Mime`, `Porygon-Z` or `Porygon2` have no such join and stay whole. Every split is printed for inspection.

In [3]:
JOIN = re.compile(r"^(.+?[a-z])([A-Z].*)$")


def split_name(s):
    m = JOIN.match(s)
    return (m.group(1), m.group(2)) if m else (s, "")


parts = df["name_raw"].map(split_name)
df["species"] = [p[0] for p in parts]
df["form"] = [p[1] for p in parts]
df["is_species"] = ~df["dex"].duplicated()                 # first row of each dex number = the species
df["form_kind"] = np.where(df.is_species, "base", np.where(df.form.str.startswith("Mega "), "mega", "alt"))
df["name"] = np.where(df.form == "", df.species,
                      np.where(df.form_kind == "mega", df.form, df.species + " (" + df.form + ")"))

# every row of one dex number must carry the same species name after the split
assert df.groupby("dex").species.nunique().max() == 1, "split produced two species names for one dex"
megas = df[df.form_kind == "mega"]
assert all(s in f for s, f in zip(megas.species, megas.form)), "a Mega label does not name its species"
assert df.is_species.sum() == 721 and df.species[df.is_species].is_unique

split_rows = df[df.form != ""]
note("split glued form labels", len(split_rows), "name_raw = species + form label with no separator")
print(split_rows.form_kind.value_counts().to_string())
print("\nbase rows that carry a forme label (species shown in its default forme):")
print(", ".join(split_rows.loc[split_rows.form_kind == "base", "name"]))
print("\nalternate forms:")
print(", ".join(split_rows.loc[split_rows.form_kind == "alt", "name"]))

[04] split glued form labels: 94 rows — name_raw = species + form label with no separator
form_kind
mega    48
alt     31
base    15

base rows that carry a forme label (species shown in its default forme):
Deoxys (Normal Forme), Wormadam (Plant Cloak), Giratina (Altered Forme), Shaymin (Land Forme), Darmanitan (Standard Mode), Tornadus (Incarnate Forme), Thundurus (Incarnate Forme), Landorus (Incarnate Forme), Keldeo (Ordinary Forme), Meloetta (Aria Forme), Meowstic (Male), Aegislash (Blade Forme), Pumpkaboo (Average Size), Gourgeist (Average Size), Hoopa (Hoopa Confined)

alternate forms:
Kyogre (Primal Kyogre), Groudon (Primal Groudon), Deoxys (Attack Forme), Deoxys (Defense Forme), Deoxys (Speed Forme), Wormadam (Sandy Cloak), Wormadam (Trash Cloak), Rotom (Heat Rotom), Rotom (Wash Rotom), Rotom (Frost Rotom), Rotom (Fan Rotom), Rotom (Mow Rotom), Giratina (Origin Forme), Shaymin (Sky Forme), Darmanitan (Zen Mode), Tornadus (Therian Forme), Thundurus (Therian Forme), Landorus (Ther

Every one of the 94 glued names splits cleanly, and each `dex` number maps to exactly one species. 15 species
appear in their default forme as the "base" row (Deoxys Normal Forme, Giratina Altered Forme, …). That is the
form the games treat as standard, so it stays as the species row.

### 2.3 `legendary_canon`

The raw flag misses eight species that the games class as legendary or mythical (`01` §1.6). The corrected
flag adds them; the raw flag stays, so results can be reported under both.

In [4]:
CANON_MISFLAGGED = ["Mew", "Celebi", "Cresselia", "Phione", "Manaphy", "Keldeo", "Meloetta", "Genesect"]
fix = df.species.isin(CANON_MISFLAGGED)
df["legendary_canon"] = df.legendary_raw | fix
note("legendary_canon = legendary_raw OR games' legendary/mythical list", int((fix & ~df.legendary_raw).sum()),
     "raw flag mixes definitions: 9 mythicals flagged, 8 legendary/mythical species not")

sp = df[df.is_species]
print(f"species: legendary_raw {sp.legendary_raw.sum()}  ->  legendary_canon {sp.legendary_canon.sum()}")
print(f"forms flagged (inherit their species): {df.loc[~df.is_species, 'legendary_canon'].sum()} of {(~df.is_species).sum()}")

[05] legendary_canon = legendary_raw OR games' legendary/mythical list: 10 rows — raw flag mixes definitions: 9 mythicals flagged, 8 legendary/mythical species not
species: legendary_raw 46  ->  legendary_canon 54
forms flagged (inherit their species): 21 of 79


### 2.4 Evolution-family proxy

There is no family column. The proxy uses the fact that evolution lines are numbered consecutively and
usually keep their primary type. **A family is a run of consecutive species (in dex order) with the same
`type1`.** It is deliberately coarse:

- It **over-merges** neighbouring lines of the same type (Pidgey, Rattata and Spearow all become one Normal
  run). That only makes the grouped split stricter.
- It **misses** lines that change primary type (Azurill Normal → Marill Water) and evolutions added in a
  later generation (Pichu #172 → Pikachu #25; Magnezone #462 → Magneton #82). Those siblings can still
  straddle a split, so a family-grouped score is, if anything, still slightly **optimistic**.

Forms inherit the family of their species.

In [5]:
order = df[df.is_species].sort_values("dex")
fam = (order.type1 != order.type1.shift()).cumsum()
df["family"] = df.dex.map(dict(zip(order.dex, fam))).astype("int64")
fsize = df[df.is_species].groupby("family").size()
note("family proxy = consecutive same-type1 run in dex order", len(df), "evolution lines are numbered consecutively; for grouped validation")
print(f"{df.family.nunique()} families for 721 species; size distribution: {fsize.value_counts().sort_index().to_dict()}")
print("\nexample families:")
for f in [1, 2, 3, 7]:
    print(f"  {f}: " + ", ".join(df.loc[(df.family == f) & df.is_species, "name"]))

[06] family proxy = consecutive same-type1 run in dex order: 800 rows — evolution lines are numbered consecutively; for grouped validation
360 families for 721 species; size distribution: {1: 142, 2: 123, 3: 67, 4: 15, 5: 7, 6: 5, 7: 1}

example families:
  1: Bulbasaur, Ivysaur, Venusaur
  2: Charmander, Charmeleon, Charizard
  3: Squirtle, Wartortle, Blastoise
  7: Pikachu, Raichu


### 2.5 Derived columns (review §7)

The most important are the six **stat shares** (stat ÷ Total). They separate the *shape* of a stat line from
its *magnitude*: a basic-stage creature and its final evolution have very different Totals but often the
same proportions. Part II of the report studies Total (magnitude) and Part III studies the shares (shape).

In [6]:
six = df[STATS].astype(float)
for s in STATS:
    df[f"sh_{s}"] = six[s] / df.total
df["total_check"] = df.total - df[STATS].sum(axis=1)
df["offence"] = df.atk + df.spa
df["bulk"] = df.hp + df["def"] + df.spd
df["phys_spec_ratio"] = df.atk / df.spa
df["off_def_ratio"] = (df.atk + df.spa) / (df["def"] + df.spd)
df["stat_sd"] = six.std(axis=1, ddof=0)
df["top_stat"] = six.idxmax(axis=1).map(STAT_LABEL)
df["low_stat"] = six.idxmin(axis=1).map(STAT_LABEL)
df["total_pct"] = df.total.rank(pct=True)
df["speed_tier"] = pd.qcut(df.spe, 4, labels=["slow", "mid-slow", "mid-fast", "fast"])
df["pseudo_600"] = df.is_species & ~df.legendary_canon & (df.total == 600)
assert (df.total_check == 0).all()
assert np.allclose(df[SHARES].sum(axis=1), 1)
note("derived columns (shares, bulk, ratios, spread, tiers, pseudo_600)", len(df), "review §7; shares split shape from magnitude")
print("pseudo-legendaries (non-legendary species at exactly 600):", ", ".join(df.loc[df.pseudo_600, "name"]))

[07] derived columns (shares, bulk, ratios, spread, tiers, pseudo_600): 800 rows — review §7; shares split shape from magnitude
pseudo-legendaries (non-legendary species at exactly 600): Dragonite, Tyranitar, Salamence, Metagross, Garchomp, Hydreigon, Goodra


### 2.6 Outliers — flagged, not removed

Extreme stats here are **design choices**, not measurement errors: Blissey's HP of 255 and Shuckle's Defense of
230 are the games' values. Removing them would change the roster being described. They are flagged
(|z| > 3 on any raw stat), and the analysis relies on rank-based tests and shares, which are robust to them.

In [7]:
z = (df[STATS] - df[STATS].mean()) / df[STATS].std()
df["extreme_stat"] = (z.abs() > 3).any(axis=1)
ext = df[df.extreme_stat & df.is_species]
note("flag |z| > 3 on any raw stat (kept)", int(df.extreme_stat.sum()), "design extremes, not errors; rank tests + shares are robust")
print(ext[["name", "type1"] + STATS].to_string(index=False))

same = df[df.is_species].duplicated(subset=["type1", "type2"] + STATS, keep=False)
print("\nspecies with an identical type and stat line:", ", ".join(df.loc[df.is_species][same].name))
note("identical stat lines kept", int(same.sum()), "three pairs of different species share one design: Charmander/Cyndaquil, Charmeleon/Quilava, Silcoon/Cascoon")

[08] flag |z| > 3 on any raw stat (kept): 32 rows — design extremes, not errors; rank tests + shares are robust
                    name   type1  hp  atk  def  spa  spd  spe
                Cloyster   Water  50   95  180   85   45   70
                 Chansey  Normal 250    5    5   35  105   50
                 Snorlax  Normal 160  110   65   65  110   30
               Wobbuffet Psychic 190   33   58   33   58   33
                 Steelix   Steel  75   85  200   55   65   30
                 Shuckle     Bug  20   10  230   10  230    5
                 Blissey  Normal 255   10   10   75  135   55
                 Slaking  Normal 150  160  100   95   65  100
                 Ninjask     Bug  61   90   45   50   50  160
                  Aggron   Steel  70  110  180   60   60   50
                 Wailord   Water 170   90   45   90   45   60
                Regirock    Rock  80  100  200   50  100   50
                  Regice     Ice  80   50  100  100  200   50
               Basti

### 2.7 Business-rule assertions and save

In [8]:
sp = df[df.is_species]
assert len(df) == 800 and len(sp) == 721
assert sp.dex.is_unique and set(sp.dex) == set(range(1, 722))
assert df[STATS].ge(1).all().all() and df[STATS].le(255).all().all()
assert (df.type1.astype(str) != df.type2.astype(str)).all()
assert df.groupby("dex").type1.nunique().max() == 1, "forms keep their species' primary type"
assert df.groupby("dex").generation.nunique().max() == 1
assert sp.legendary_canon.sum() == 54

cols = ["dex", "name", "species", "form", "form_kind", "is_species", "name_raw", "type1", "type2", "is_dual_type",
        "type_combo", "generation", "legendary_raw", "legendary_canon", "pseudo_600", "family", "total"] + STATS + \
       SHARES + ["offence", "bulk", "phys_spec_ratio", "off_def_ratio", "stat_sd", "top_stat", "low_stat",
                 "total_pct", "speed_tier", "extreme_stat", "total_check"]
df = df[cols]
df.to_parquet(PROC / "pokemon_clean.parquet", index=False)
pd.DataFrame(log).to_csv(PROC / "_cleaning_log.csv", index=False)
print(f"saved data/processed/pokemon_clean.parquet  {df.shape}")
pd.DataFrame(log)

saved data/processed/pokemon_clean.parquet  (800, 40)


,step,rows_affected,why
0,rename to snake_case,800,"stable, quote-free column names"
1,type1/type2 -> 18-level categorical,800,"fixed order, all levels present in every group-by"
2,type2 left null where single-typed,386,"structural absence (one type), not missing dat..."
3,split glued form labels,94,name_raw = species + form label with no separator
4,legendary_canon = legendary_raw OR games' lege...,10,raw flag mixes definitions: 9 mythicals flagge...
5,family proxy = consecutive same-type1 run in d...,800,evolution lines are numbered consecutively; fo...
6,"derived columns (shares, bulk, ratios, spread,...",800,review §7; shares split shape from magnitude
7,flag |z| > 3 on any raw stat (kept),32,"design extremes, not errors; rank tests + shar..."
8,identical stat lines kept,6,three pairs of different species share one des...


In [9]:
kf_update(
    n_glued_names=int(len(split_rows)), n_families=int(df.family.nunique()),
    family_size_max=int(fsize.max()), n_legendary_canon=int(sp.legendary_canon.sum()),
    n_pseudo_600=int(sp.pseudo_600.sum()), pseudo_names=list(sp.loc[sp.pseudo_600, "name"]),
    n_extreme_species=int(len(ext)), n_dual_species=int(sp.is_dual_type.sum()),
    n_cleaning_steps=len(log),
)

  _key_figures.json <- n_glued_names, n_families, family_size_max, n_legendary_canon, n_pseudo_600, pseudo_names, n_extreme_species, n_dual_species, n_cleaning_steps  (33 figures total)


## Stage 2 — Gate Checklist

- [x] Missing-value strategy per column: only `type2` is null; a structural absence, flagged, not imputed
- [x] Duplicates assessed: no duplicate rows; 79 forms share a `dex` with their species and are flagged
      (`form_kind`), not dropped; three pairs of species with identical stat lines (Charmander/Cyndaquil, Charmeleon/Quilava,
      Silcoon/Cascoon) kept: they are distinct species the designers gave the same numbers
- [x] All columns have correct dtypes (18-level categoricals for types)
- [x] Business-rule validations pass (assertions above)
- [x] Cleaning log exists (`data/processed/_cleaning_log.csv`)
- [x] Cleaned data saved to `data/processed/pokemon_clean.parquet`

**Changelog, pass 2 (after `03`):** `pseudo_600` was added once the Total histogram showed a spike of
non-legendaries at exactly 600; `03` §3.2 shows it.